# Phân tích Dự báo Nhu cầu & Tối ưu hóa Tồn kho — Phase 2

## Executive Summary

Notebook này thực hiện toàn bộ Giai đoạn 2 của dự án DataCo Supply Chain Analytics, bao gồm:

1. **EDA Chuỗi thời gian**: Phân tích xu hướng nhu cầu theo tháng cho Top Category và Top Products.
2. **Seasonal Decomposition**: Tách biệt Trend, Seasonality và Residual từ dữ liệu lịch sử.
3. **Demand Forecasting**: Dự báo ngắn hạn (3 tháng - MA3) và dài hạn (12 tháng - Linear Regression).
4. **Newsvendor Model**: Tính lượng đặt hàng tối ưu Q* dưới điều kiện bất định, giả định Holding Cost cao.
5. **Monte Carlo Simulation**: Xác nhận Q* bằng 10,000 lần mô phỏng biến động nhu cầu.

---

**Dữ liệu**: DataCoSupplyChainDataset.csv (180,519 giao dịch, 53 cột) — mount từ Google Drive.

**Framework áp dụng**: Operations Analytics (Newsvendor), Monte Carlo Simulation, Time-Series Forecasting.


## Bước 0: Mount Google Drive
Kết nối Google Drive để truy cập dataset. Chỉ cần chạy 1 lần mỗi session.

In [ ]:
# ============================================================
# BƯỚC 0: Mount Google Drive
# Chỉ cần chạy 1 lần, sau đó bấm 'Connect to Google Drive'
# trong hộp thoại xác nhận quyền.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive đã được mount thành công!')


## Bước 1: Import Thư viện
Import tất cả thư viện cần thiết cho phân tích.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12

print('✅ Tất cả thư viện đã import thành công.')


## Bước 2: Load & Clean Data

Đọc dataset từ Google Drive, loại bỏ đơn hàng bị hủy (`CANCELED`) và nghi ngờ gian lận (`SUSPECTED_FRAUD`),
sau đó parse ngày tháng và tạo các cột thời gian phục vụ phân tích.

In [ ]:
# Đường dẫn dataset trên Google Drive
data_path = '/content/drive/MyDrive/DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS/DataCoSupplyChainDataset.csv'

df = pd.read_csv(data_path, encoding='ISO-8859-1', low_memory=False)

# Loại bỏ đơn hủy & gian lận
df_clean = df[~df['Order Status'].isin(['CANCELED', 'SUSPECTED_FRAUD'])].copy()

# Parse ngày tháng
df_clean['Order_Date'] = pd.to_datetime(df_clean['order date (DateOrders)'], errors='coerce')
df_clean = df_clean.dropna(subset=['Order_Date'])

# Tạo cột thời gian
df_clean['Year']      = df_clean['Order_Date'].dt.year
df_clean['Month']     = df_clean['Order_Date'].dt.month
df_clean['YearMonth'] = df_clean['Order_Date'].dt.to_period('M')

print(f'Shape sau cleaning: {df_clean.shape}')
print(f'Khoảng thời gian  : {df_clean["Order_Date"].min().date()} → {df_clean["Order_Date"].max().date()}')
print(f'Số Category       : {df_clean["Category Name"].nunique()}')
print(f'Số Product        : {df_clean["Product Name"].nunique()}')


## Bước 3: EDA — Phân tích Nhu cầu theo Thời gian

Xác định Top 3 Category và Top 5 Products theo tổng số lượng đặt hàng,
sau đó vẽ biểu đồ xu hướng nhu cầu theo tháng.

In [ ]:
# Top 3 Category theo tổng số lượng đặt hàng
top_categories = (df_clean.groupby('Category Name')['Order Item Quantity']
                  .sum().nlargest(3).index.tolist())
print('Top 3 Category:', top_categories)

# Top 5 sản phẩm theo tổng số lượng
top_products = (df_clean.groupby('Product Name')['Order Item Quantity']
                .sum().nlargest(5).index.tolist())
print('Top 5 Products:', top_products)


In [ ]:
# Biểu đồ tổng nhu cầu toàn công ty theo tháng
monthly_total = (df_clean.groupby('YearMonth')['Order Item Quantity']
                 .sum().reset_index())
monthly_total['YearMonth_dt'] = monthly_total['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_total['YearMonth_dt'], monthly_total['Order Item Quantity'],
        marker='o', linewidth=2, color='steelblue')
ax.fill_between(monthly_total['YearMonth_dt'], monthly_total['Order Item Quantity'],
                alpha=0.15, color='steelblue')
ax.set_title('Tổng Nhu cầu Đặt hàng theo Tháng — Toàn Công ty', fontsize=14, fontweight='bold')
ax.set_xlabel('Tháng')
ax.set_ylabel('Số lượng đặt hàng')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Biểu đồ nhu cầu theo tháng cho Top 3 Category
fig, axes = plt.subplots(len(top_categories), 1, figsize=(14, 5 * len(top_categories)))
if len(top_categories) == 1:
    axes = [axes]

palette = ['steelblue', 'darkorange', 'seagreen']
for ax, cat, color in zip(axes, top_categories, palette):
    cat_monthly = (df_clean[df_clean['Category Name'] == cat]
                   .groupby('YearMonth')['Order Item Quantity']
                   .sum().reset_index())
    cat_monthly['YearMonth_dt'] = cat_monthly['YearMonth'].dt.to_timestamp()

    ax.plot(cat_monthly['YearMonth_dt'], cat_monthly['Order Item Quantity'],
            marker='o', linewidth=2, color=color)
    ax.fill_between(cat_monthly['YearMonth_dt'], cat_monthly['Order Item Quantity'],
                    alpha=0.15, color=color)
    ax.set_title(f'Nhu cầu theo tháng — {cat}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Số lượng')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()


## Bước 4: Phân tách Mùa vụ (Seasonal Decomposition)

Tách chuỗi nhu cầu thành 3 thành phần:
- **Trend**: Xu hướng dài hạn (tăng/giảm)
- **Seasonality**: Biến động chu kỳ 12 tháng (mùa vụ)
- **Residual**: Nhiễu ngẫu nhiên còn lại

Dùng `seasonal_decompose` với `model='additive'` (phù hợp khi biên độ mùa vụ không đổi theo thời gian).

In [ ]:
for cat in top_categories:
    cat_monthly = (df_clean[df_clean['Category Name'] == cat]
                   .groupby('YearMonth')['Order Item Quantity']
                   .sum())
    cat_monthly.index = cat_monthly.index.to_timestamp()
    cat_monthly = cat_monthly.asfreq('MS').fillna(0)

    if len(cat_monthly) < 24:
        print(f'[SKIP] {cat}: không đủ dữ liệu (cần >=24 tháng, có {len(cat_monthly)})')
        continue

    result = seasonal_decompose(cat_monthly, model='additive', period=12)

    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    result.observed.plot(ax=axes[0], title='Observed (Thực tế)', color='steelblue')
    result.trend.plot(ax=axes[1], title='Trend (Xu hướng)', color='orange')
    result.seasonal.plot(ax=axes[2], title='Seasonality (Mùa vụ)', color='green')
    result.resid.plot(ax=axes[3], title='Residual (Nhiễu)', color='red')
    for a in axes:
        a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.suptitle(f'Seasonal Decomposition — {cat}', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # Tháng cao điểm
    seasonal_df = result.seasonal.reset_index()
    seasonal_df.columns = ['Date', 'Seasonal']
    seasonal_df['Month'] = seasonal_df['Date'].dt.month
    monthly_season = seasonal_df.groupby('Month')['Seasonal'].mean()
    peak_month = monthly_season.idxmax()
    low_month  = monthly_season.idxmin()
    print(f'  [{cat}] Tháng cao điểm: Tháng {peak_month}  |  Tháng thấp điểm: Tháng {low_month}')


## Bước 5: Dự báo Nhu cầu (Demand Forecasting)

**Ngắn hạn (3 tháng)**: Moving Average 3 tháng (MA3) × Seasonal Index — phù hợp vận hành.

**Dài hạn (12 tháng)**: Linear Regression trên trend × Seasonal Index — phù hợp lập kế hoạch.

Áp dụng cho **Top 3 Category**, sau đó **drill-down** vào **Top 5 Products**.

In [ ]:
def forecast_category(df_input, category, forecast_months_short=3, forecast_months_long=12):
    cat_data = (df_input[df_input['Category Name'] == category]
                .groupby('YearMonth')['Order Item Quantity']
                .sum())
    cat_data.index = cat_data.index.to_timestamp()
    cat_data = cat_data.asfreq('MS').fillna(0)

    # --- Moving Average 3 tháng ---
    ma3 = cat_data.rolling(window=3).mean()
    last_3_avg = cat_data[-3:].mean()

    # Seasonal index: tỉ lệ tháng so với trung bình năm
    seasonal_idx = cat_data.groupby(cat_data.index.month).mean() / cat_data.mean()

    last_date = cat_data.index[-1]
    short_dates = pd.date_range(start=last_date + pd.offsets.MonthBegin(1),
                                periods=forecast_months_short, freq='MS')
    short_forecast = pd.Series(
        [last_3_avg * seasonal_idx.get(d.month, 1) for d in short_dates],
        index=short_dates
    )

    # --- Linear Regression dài hạn ---
    X = np.arange(len(cat_data)).reshape(-1, 1)
    y = cat_data.values
    model = LinearRegression().fit(X, y)

    long_X = np.arange(len(cat_data), len(cat_data) + forecast_months_long).reshape(-1, 1)
    long_dates = pd.date_range(start=last_date + pd.offsets.MonthBegin(1),
                               periods=forecast_months_long, freq='MS')
    long_base = model.predict(long_X)
    long_forecast = pd.Series(
        [max(0, long_base[i] * seasonal_idx.get(d.month, 1)) for i, d in enumerate(long_dates)],
        index=long_dates
    )

    return {
        'series': cat_data, 'ma3': ma3,
        'short_forecast': short_forecast,
        'long_forecast': long_forecast,
        'seasonal_idx': seasonal_idx, 'model': model
    }

print('✅ Hàm forecast_category() đã sẵn sàng.')


In [ ]:
forecast_results = {}
palette = ['steelblue', 'darkorange', 'seagreen']

for cat, color in zip(top_categories, palette):
    res = forecast_category(df_clean, cat)
    forecast_results[cat] = res

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(res['series'].index, res['series'].values,
            label='Lịch sử thực tế', color=color, linewidth=2)
    ax.plot(res['ma3'].index, res['ma3'].values,
            label='MA3 (smoothed)', color='gray', linestyle='--', linewidth=1.5)
    ax.plot(res['short_forecast'].index, res['short_forecast'].values,
            label='Dự báo ngắn hạn 3T (MA)', color='green',
            marker='o', linewidth=2, linestyle='-.')
    ax.plot(res['long_forecast'].index, res['long_forecast'].values,
            label='Dự báo dài hạn 12T (LR)', color='red',
            marker='s', linewidth=2, linestyle=':')
    ax.axvline(x=res['series'].index[-1], color='gray', linestyle='--',
               alpha=0.5, label='Thời điểm hiện tại')
    ax.set_title(f'Dự báo Nhu cầu — {cat}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tháng')
    ax.set_ylabel('Số lượng đặt hàng')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend(loc='upper left', fontsize=9)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print(f'\n[{cat}] Dự báo ngắn hạn (MA3 × Seasonal):')
    for d, v in res['short_forecast'].items():
        print(f'  {d.strftime("%Y-%m")}: {v:,.0f} đơn vị')
    print(f'[{cat}] Dự báo dài hạn (12 tháng - LR):')
    for d, v in res['long_forecast'].items():
        print(f'  {d.strftime("%Y-%m")}: {v:,.0f} đơn vị')


### Drill-down: Dự báo Top 5 Sản phẩm

In [ ]:
def forecast_product(df_input, product_name, forecast_months=3):
    prod_data = (df_input[df_input['Product Name'] == product_name]
                 .groupby('YearMonth')['Order Item Quantity']
                 .sum())
    prod_data.index = prod_data.index.to_timestamp()
    prod_data = prod_data.asfreq('MS').fillna(0)
    ma3 = prod_data.rolling(window=3).mean()
    last_3_avg = prod_data[-3:].mean()
    last_date  = prod_data.index[-1]
    forecast_dates = pd.date_range(start=last_date + pd.offsets.MonthBegin(1),
                                   periods=forecast_months, freq='MS')
    seasonal_idx = prod_data.groupby(prod_data.index.month).mean() / prod_data.mean()
    fc_vals = pd.Series(
        [last_3_avg * seasonal_idx.get(d.month, 1) for d in forecast_dates],
        index=forecast_dates
    )
    return prod_data, ma3, fc_vals

fig, axes = plt.subplots(len(top_products), 1, figsize=(14, 4 * len(top_products)))
if len(top_products) == 1:
    axes = [axes]

for ax, prod in zip(axes, top_products):
    series, ma3, fc = forecast_product(df_clean, prod)
    ax.plot(series.index, series.values, label='Thực tế', color='steelblue', linewidth=1.5)
    ax.plot(ma3.index,    ma3.values,    label='MA3',     color='orange', linestyle='--')
    ax.plot(fc.index,     fc.values,     label='Dự báo 3T', color='green', marker='o', linewidth=2)
    ax.axvline(x=series.index[-1], color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'Product: {prod}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Số lượng')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.suptitle('Dự báo Nhu cầu — Top 5 Sản phẩm', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## Bước 6: Newsvendor Model — Tính Lượng Đặt hàng Tối ưu Q*

### Lý thuyết Newsvendor

| Ký hiệu | Ý nghĩa | Nguồn tính |
|---------|---------|------------|
| **Cu** | Underage / Stockout Cost — lợi nhuận bị mất khi thiếu hàng | `Benefit per order` (mean) |
| **Co** | Overage / Holding Cost — chi phí lưu kho khi thừa hàng | `Product Price × 20%` |
| **CR** | Critical Ratio = Cu / (Cu + Co) | Tính từ Cu, Co |
| **Q*** | Phân vị CR của phân phối Normal(μ, σ) | `scipy.stats.norm.ppf(CR, μ, σ)` |

> **Lưu ý**: Vì Holding Cost cao → CR < 0.5 → Q* < Mean Demand.
> Điều này có nghĩa: **chấp nhận rủi ro hết hàng nhỏ để tránh chi phí tồn kho lớn**.


In [ ]:
newsvendor_results = []

for cat in top_categories:
    # Nhu cầu lịch sử theo tháng
    cat_monthly = (df_clean[df_clean['Category Name'] == cat]
                   .groupby('YearMonth')['Order Item Quantity']
                   .sum())
    mu    = cat_monthly.mean()
    sigma = cat_monthly.std()

    # Chi phí
    cat_df    = df_clean[df_clean['Category Name'] == cat]
    cu        = cat_df['Benefit per order'].mean()           # Stockout cost
    avg_price = cat_df['Order Item Product Price'].mean()
    co        = avg_price * 0.20                             # Holding cost = 20% giá sản phẩm

    cr     = cu / (cu + co)                                  # Critical Ratio
    q_star = stats.norm.ppf(cr, loc=mu, scale=sigma)         # Q* từ phân vị CR
    q_star = max(0, round(q_star))

    newsvendor_results.append({
        'Category': cat,
        'Mean Demand μ': round(mu, 1),
        'Std Dev σ': round(sigma, 1),
        'Cu Stockout $': round(cu, 2),
        'Co Holding $': round(co, 2),
        'Critical Ratio': round(cr, 4),
        'Q* (đv/tháng)': q_star
    })

    print(f"\n{'='*60}")
    print(f'Category: {cat}')
    print(f'  μ (mean demand/tháng) = {mu:,.1f}')
    print(f'  σ (std dev)           = {sigma:,.1f}')
    print(f'  Cu (Stockout)  = ${cu:.2f}  |  Co (Holding) = ${co:.2f}')
    print(f'  Critical Ratio = {cr:.4f}  →  Q* = {q_star:,} đơn vị/tháng')
    if cr < 0.5:
        print(f'  ⚠️  CR < 0.5 → Holding Cost cao → đặt ÍT HƠN trung bình ({mu:,.0f})')
    else:
        print(f'  ✅  CR > 0.5 → Stockout Cost cao → đặt NHIỀU HƠN trung bình ({mu:,.0f})')

nv_df = pd.DataFrame(newsvendor_results)
display(nv_df)


In [ ]:
# Biểu đồ so sánh Q* với Mean Demand
fig, ax = plt.subplots(figsize=(10, 5))
x      = np.arange(len(newsvendor_results))
width  = 0.35
means  = [r['Mean Demand μ'] for r in newsvendor_results]
qstars = [r['Q* (đv/tháng)'] for r in newsvendor_results]
cats_  = [r['Category'] for r in newsvendor_results]

bars1 = ax.bar(x - width/2, means,  width, label='Mean Demand (μ)', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, qstars, width, label='Q* Tối ưu',       color='darkorange', alpha=0.8)
ax.bar_label(bars1, fmt='%.0f', padding=3)
ax.bar_label(bars2, fmt='%.0f', padding=3)
ax.set_xticks(x)
ax.set_xticklabels(cats_, rotation=15, ha='right')
ax.set_ylabel('Số lượng đặt hàng / tháng')
ax.set_title('So sánh Q* Tối ưu vs Mean Demand theo Category', fontsize=13, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()


## Bước 7: Monte Carlo Simulation — Mô phỏng Rủi ro Tồn kho

Chạy **10,000 lần mô phỏng** biến động nhu cầu theo phân phối Normal(μ, σ) cho từng Category.

Với mỗi mức Q được thử, tính tổng chi phí kỳ vọng:
- Nếu **Demand > Q**: Chi phí thiếu hàng = Cu × (Demand − Q)
- Nếu **Demand < Q**: Chi phí lưu kho = Co × (Q − Demand)

Q_mc* là mức Q cho **tổng chi phí kỳ vọng thấp nhất** — dùng để cross-validate Q* Newsvendor.

> ✅ Kết quả được coi là hợp lệ nếu |Q_mc* − Q_nv*| < 10%

In [ ]:
np.random.seed(42)
N_SIMULATIONS = 10_000
mc_results_all = {}

for r in newsvendor_results:
    cat       = r['Category']
    mu        = r['Mean Demand μ']
    sigma     = r['Std Dev σ']
    cu        = r['Cu Stockout $']
    co        = r['Co Holding $']
    q_star_nv = r['Q* (đv/tháng)']

    # Dải Q cần thử: μ ± 2σ
    q_range = np.arange(max(0, int(mu - 2*sigma)),
                        int(mu + 2*sigma) + 1,
                        max(1, int(sigma / 5)))

    # Mô phỏng nhu cầu
    demands = np.maximum(np.random.normal(loc=mu, scale=sigma, size=N_SIMULATIONS), 0)

    # Tính chi phí kỳ vọng cho từng mức Q
    expected_costs = []
    for q in q_range:
        overstock  = np.maximum(q - demands, 0)
        understock = np.maximum(demands - q, 0)
        expected_costs.append((co * overstock + cu * understock).mean())

    q_mc_star = int(q_range[np.argmin(expected_costs)])
    mc_results_all[cat] = {
        'q_range': q_range, 'expected_costs': expected_costs,
        'q_mc_star': q_mc_star, 'q_star_nv': q_star_nv, 'demands': demands
    }

    # ── Biểu đồ ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Cost curve
    axes[0].plot(q_range, expected_costs, color='steelblue', linewidth=2)
    axes[0].axvline(x=q_mc_star, color='darkorange', linestyle='--', linewidth=2,
                    label=f'Q* MC = {q_mc_star:,}')
    axes[0].axvline(x=q_star_nv, color='green',      linestyle=':',  linewidth=2,
                    label=f'Q* Newsvendor = {q_star_nv:,}')
    axes[0].set_title(f'[{cat}] Chi phí Kỳ vọng theo Q', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Lượng đặt hàng Q')
    axes[0].set_ylabel('Chi phí kỳ vọng ($)')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    axes[0].legend()

    # Phân phối nhu cầu MC
    axes[1].hist(demands, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
    axes[1].axvline(x=q_mc_star, color='darkorange', linestyle='--', linewidth=2,
                    label=f'Q* MC = {q_mc_star:,}')
    axes[1].axvline(x=mu, color='gray', linestyle=':', linewidth=1.5,
                    label=f'Mean μ = {mu:,.0f}')
    axes[1].set_title(f'[{cat}] Phân phối Nhu cầu MC (n={N_SIMULATIONS:,})', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Nhu cầu (đơn vị)')
    axes[1].set_ylabel('Tần suất')
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    diff_pct = abs(q_mc_star - q_star_nv) / max(q_star_nv, 1) * 100
    status   = '✅ Hợp lệ' if diff_pct < 10 else '⚠️ Chênh lệch lớn — cần xem lại'
    print(f'[{cat}] Q* Newsvendor = {q_star_nv:,}  |  Q* Monte Carlo = {q_mc_star:,}')
    print(f'  Chênh lệch: {diff_pct:.1f}%  →  {status}')


## Bước 8: Executive Summary & Khuyến nghị Hành động

In [ ]:
print()
print('=' * 70)
print('EXECUTIVE SUMMARY — DEMAND FORECASTING & INVENTORY OPTIMIZATION')
print('=' * 70)

for cat in top_categories:
    r  = next(x for x in newsvendor_results if x['Category'] == cat)
    mc = mc_results_all[cat]
    res = forecast_results[cat]

    print(f'\n📦 Category: {cat}')
    print(f'   Nhu cầu trung bình/tháng : {r["Mean Demand μ"]:>10,.1f} đơn vị')
    print(f'   Độ lệch chuẩn (σ)        : {r["Std Dev σ"]:>10,.1f} đơn vị')
    print(f'   Q* Newsvendor Formula    : {r["Q* (đv/tháng)"]:>10,} đơn vị/tháng')
    print(f'   Q* Monte Carlo Confirmed : {mc["q_mc_star"]:>10,} đơn vị/tháng')
    print(f'   Critical Ratio           : {r["Critical Ratio"]:>10.4f}')

    # Dự báo 3 tháng tới
    print(f'   Dự báo nhu cầu 3T tới   :')
    for d, v in res['short_forecast'].items():
        print(f'     {d.strftime("%Y-%m")}: {v:,.0f} đơn vị')

    if r['Critical Ratio'] < 0.5:
        print(f'   ⚠️  CR < 0.5 → Holding Cost cao → nên đặt ÍT HƠN mean demand')
    else:
        print(f'   ✅  CR > 0.5 → Stockout Cost cao → nên đặt NHIỀU HƠN mean demand')


## Khuyến nghị Hành động

1. **Đặt hàng theo Q***: Mỗi tháng đặt hàng theo Q* đã tính từ Newsvendor Model.
   - Q* đã được xác nhận bởi Monte Carlo → có căn cứ định lượng vững chắc.
   - Q* < Mean Demand phản ánh Holding Cost cao — chấp nhận rủi ro hết hàng nhỏ để tránh chi phí lưu kho lớn.

2. **Điều chỉnh theo mùa vụ**: Tăng Q lên ~15–20% so với Q* trong các **tháng cao điểm** theo Seasonal Index.
   Giảm Q xuống ~10–15% trong **tháng thấp điểm**.

3. **Review định kỳ hàng quý**: Cập nhật μ, σ, Cu từ dữ liệu thực mới nhất để Q* luôn phản ánh thực tế vận hành.

4. **Ghi nhận Lost Sales (Censored Demand)**: Khi hết hàng, doanh số (Sales) thấp hơn nhu cầu thực (Demand).
   Cần bổ sung cơ chế theo dõi lost sales để phân tích Censored Demand và cải thiện độ chính xác mô hình.

---

## 🔍 Góc nhìn Phản biện

- **Giả định Normal Distribution**: Mô hình giả định nhu cầu phân phối chuẩn. Nếu dữ liệu thực có skewness lớn
  (hàng theo mùa mạnh), cần dùng phân phối phù hợp hơn (Poisson, Log-Normal).
- **Holding Cost 20%**: Đây là ước tính tiêu chuẩn trong ngành. Cần xác nhận với bộ phận tài chính để dùng con số thực tế.
- **Stockout Cost = Benefit per order**: Giả định mỗi đơn thiếu hàng mất toàn bộ lợi nhuận. Trên thực tế,
  khách hàng có thể backorder hoặc chuyển sang sản phẩm khác.
- **Monte Carlo 10,000 simulations**: Cho kết quả ổn định (~95% CI), có thể tăng lên 100,000 nếu cần độ chính xác cao hơn.
